# Проект 2. Кодовое шифрование: учебная схема Мак-Элиса

**Выполнил:** Максим Бердников  
**Цель:** Реализовать учебную версию схемы Мак-Элиса на коде Хэмминга [7, 4, 3] и продемонстрировать атаку brute-force.

## 1. Арифметика над GF(2)

Реализация операций над векторами и матрицами по модулю 2:
- Сложение по модулю 2;
- Умножение матриц по модулю 2;
- Обращение бинарных матриц.

In [ ]:
import time
import numpy as np


def gf2_add(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """
    Сложить два вектора или матрицы над GF(2).
    """
    a = np.asarray(a, dtype=np.int8)
    b = np.asarray(b, dtype=np.int8)
    return (a + b) % 2


def gf2_mul(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """
    Перемножить две матрицы (или матрицу и вектор) над GF(2).
    """
    a = np.asarray(a, dtype=np.int8)
    b = np.asarray(b, dtype=np.int8)
    return np.mod(a @ b, 2)


def gf2_inv(matrix: np.ndarray) -> np.ndarray:
    """
    Вычислить обратную матрицу над GF(2) методом Гаусса-Жордана.
    Возвращает матрицу или выбрасывает ValueError, если матрица вырождена.
    """
    a = np.asarray(matrix, dtype=np.int8) % 2
    if a.ndim != 2 or a.shape[0] != a.shape[1]:
        raise ValueError("Матрица должна быть двумерной и квадратной.")
    n = a.shape[0]
    identity = np.eye(n, dtype=np.int8)
    aug = np.block([a, identity])
    for i in range(n):
        pivot_row = i
        while pivot_row < n and aug[pivot_row, i] == 0:
            pivot_row += 1
        if pivot_row == n:
            raise ValueError("Матрица вырождена (необратима).")
        if pivot_row != i:
            aug[[i, pivot_row]] = aug[[pivot_row, i]]
        for j in range(n):
            if j != i and aug[j, i] == 1:
                aug[j] = aug[j] ^ aug[i]
    return aug[:, n:]


print("Арифметика GF(2) определена: gf2_add, gf2_mul, gf2_inv")

## 2. Код Хэмминга [7, 4, 3]

Порождающая матрица $G$ (размер 4×7) в систематическом виде $[I_4 | P]$
и проверочная матрица $H$ (размер 3×7) в виде $[P^T | I_3]$.

In [ ]:
# Порождающая матрица G (размер 4x7) в систематическом виде: [I_4 | P]
# I_4 - единичная матрица 4x4, P - матрица чётности 4x3.
G: np.ndarray = np.array(
    [
        [1, 0, 0, 0, 1, 1, 0],
        [0, 1, 0, 0, 1, 0, 1],
        [0, 0, 1, 0, 0, 1, 1],
        [0, 0, 0, 1, 1, 1, 1],
    ],
    dtype=np.int8,
)

# Проверочная матрица H (размер 3x7) в виде: [P^T | I_3]
# P^T - транспонированная матрица чётности, I_3 - единичная матрица 3x3.
H: np.ndarray = np.array(
    [
        [1, 1, 0, 1, 1, 0, 0],
        [1, 0, 1, 1, 0, 1, 0],
        [0, 1, 1, 1, 0, 0, 1],
    ],
    dtype=np.int8,
)


def encode(message: np.ndarray) -> np.ndarray:
    """Кодирует 4-битное сообщение в 7-битное кодовое слово.

    Принимает вектор-строку длины 4 и умножает её на порождающую матрицу G
    над полем GF(2).
    """
    message = np.asarray(message, dtype=np.int8)
    assert message.ndim == 1, "Сообщение должно быть одномерным вектором."
    assert len(message) == 4, "Сообщение должно иметь длину 4."
    msg_2d = message[np.newaxis, :]
    code_2d = gf2_mul(msg_2d, G)
    return code_2d[0]


def syndrome(codeword: np.ndarray) -> np.ndarray:
    """Вычисляет синдром для 7-битного принятого слова.

    Формула синдрома: s = (r * H^T) % 2.
    Если синдром состоит из одних нулей, ошибок в кодовом слове нет.
    """
    codeword = np.asarray(codeword, dtype=np.int8)
    assert codeword.ndim == 1, "Кодовое слово должно быть одномерным вектором."
    assert len(codeword) == 7, "Кодовое слово должно иметь длину 7."
    code_2d = codeword[np.newaxis, :]
    syn_2d = gf2_mul(code_2d, H.T)
    return syn_2d[0]


def correct(codeword: np.ndarray, syndrome_table: dict) -> np.ndarray:
    """Исправление одиночной ошибки в кодовом слове по таблице синдромов.

    Возвращает исправленное 7-битное кодовое слово.
    """
    codeword_copy = codeword.copy()
    s = syndrome(codeword_copy)
    s_tuple = tuple(s)
    if s_tuple not in syndrome_table:
        raise ValueError(f"Неизвестный синдром {list(s)} — возможно более 1 ошибки.")
    pos = syndrome_table[s_tuple]
    if pos != -1:
        codeword_copy[pos] = (codeword_copy[pos] + 1) % 2
    return codeword_copy


print("Порождающая матрица G:\n", G)
print("\nПроверочная матрица H:\n", H)

# Проверка ортогональности: G * H^T = 0 (mod 2)
check = gf2_mul(G, H.T)
print("\nПроверка G * H^T = 0 (mod 2):\n", check)

## 3. Таблица синдромов

Построение таблицы синдромов для исправления одиночной ошибки.

**Формула синдрома:** $\text{syndrome} = r \cdot H^T \pmod{2}$

При одиночной ошибке в позиции $j$ синдром равен $j$-му столбцу матрицы $H$.

In [ ]:
def build_syndrome_table() -> dict:
    """Строит таблицу синдромов для исправления одиночных ошибок.

    Ключ словаря — кортеж значений синдрома (например, (1, 0, 1)).
    Значение — индекс бита в кодовом слове (0-6), в котором произошла ошибка.
    Для нулевого синдрома (ошибок нет) возвращает индекс -1.
    """
    table: dict = {}
    zero_syndrome = tuple(np.zeros(3, dtype=np.int8))
    table[zero_syndrome] = -1
    for pos in range(7):
        e = np.zeros(7, dtype=np.int8)
        e[pos] = 1
        s = syndrome(e)
        table[tuple(s)] = pos
    return table


syndrome_table = build_syndrome_table()

for syn, error_pos in syndrome_table.items():
    if error_pos == -1:
        print(f"Синдром {syn} -> Ошибок нет")
    else:
        print(f"Синдром {syn} -> Ошибка в бите {error_pos}")

## 4. Реализация схемы Мак-Элиса

- **Генерация ключей (`keygen`)**: случайная обратимая $S$ ($4\times4$) и перестановочная $P$ ($7\times7$), открытый ключ $G' = S \cdot G \cdot P$.
- **Шифрование (`encrypt`)**: сообщение $m$ + случайный вектор ошибки $e$ веса 1:
  $$c = m \cdot G' + e \pmod{2}$$
- **Расшифрование (`decrypt`)**: обратная перестановка $P^{-1}$, исправление по синдрому, снятие маскирования $S^{-1}$.

In [ ]:
def keygen() -> tuple:
    """Генерация ключей для схемы Мак-Элиса.

    Returns:
        pk: открытый ключ (g_pub)
        sk: словарь приватных ключей (S, P, H, S_inv, P_inv)
    """
    # 1. Генерация случайной обратимой матрицы S размера 4x4
    while True:
        s = np.random.randint(0, 2, size=(4, 4), dtype=np.int8)
        try:
            s_inv = gf2_inv(s)
            break
        except ValueError:
            continue

    # 2. Генерация случайной перестановочной матрицы P размера 7x7
    perm = np.random.permutation(7)
    p = np.zeros((7, 7), dtype=np.int8)
    for i in range(7):
        p[i, perm[i]] = 1
    p_inv = p.T  # Обратная перестановочная матрица равна её транспонированной версии

    # 3. Вычисление публичного ключа g_pub = S * G * P (mod 2)
    tmp = gf2_mul(s, G)
    g_pub = gf2_mul(tmp, p)

    sk = {"S": s, "P": p, "H": H, "S_inv": s_inv, "P_inv": p_inv}
    return g_pub, sk


def encrypt(m: np.ndarray, pk: np.ndarray) -> np.ndarray:
    """Шифрование сообщения (4 бита) плюс добавление случайной ошибки веса 1.

    Args:
        m: вектор сообщения длины 4
        pk: публичный ключ g_pub (размера 4x7)
    """
    m = np.asarray(m, dtype=np.int8)
    pk = np.asarray(pk, dtype=np.int8)
    # Генерация ошибки e веса 1
    e = np.zeros(7, dtype=np.int8)
    err_pos = np.random.randint(0, 7)
    e[err_pos] = 1
    return gf2_add(gf2_mul(m, pk), e)


def decrypt(c: np.ndarray, sk: dict) -> np.ndarray:
    """Декодирование шифртекста при помощи приватных ключей.

    Args:
        c: шифртекст (вектор длины 7)
        sk: словарь приватных ключей
    """
    c = np.asarray(c, dtype=np.int8)
    p_inv = sk["P_inv"]
    s_inv = sk["S_inv"]

    # 1. Применить обратную перестановку
    c_prime = gf2_mul(c, p_inv)

    # 2. Вычислить синдром и исправить ошибку
    syn_table = build_syndrome_table()
    c_prime_corrected = correct(c_prime, syn_table)

    # 3. Извлечь систематическую часть сообщения m' = m * S (первые 4 бита)
    m_prime = c_prime_corrected[:4]

    # 4. Снять маскирование исходного сообщения m = m' * S_inv
    return gf2_mul(m_prime, s_inv)


def brute_force(c: np.ndarray, pk: np.ndarray) -> tuple:
    """Атака полного перебора на шифртекст по открытому ключу.

    Args:
        c: шифртекст (вектор длины 7)
        pk: публичный ключ g_pub (размера 4x7)

    Returns:
        Кортеж (m, e), где m - исходное сообщение, e - вектор ошибки
    """
    c = np.asarray(c, dtype=np.int8)
    pk = np.asarray(pk, dtype=np.int8)

    for i in range(16):
        # Двоичное представление i в виде вектора длины 4
        m = np.array([(i >> 3) & 1, (i >> 2) & 1, (i >> 1) & 1, i & 1], dtype=np.int8)
        # Вычисляем e_cand = (c - m * pk) % 2
        e_cand = gf2_add(c, gf2_mul(m, pk))
        # Если вес Хэмминга равен 1, то мы нашли m и e
        if np.sum(e_cand) == 1:
            return m, e_cand

    raise ValueError("Сообщение не найдено. Возможно, добавлено более одной ошибки.")


print("Функции keygen, encrypt, decrypt, brute_force определены.")

# Демонстрация: показать пример ключей
pk_demo, sk_demo = keygen()
print("\nПример открытого ключа G' (4x7):")
print(pk_demo)
print("\nМатрица S (скрэмблер 4x4):")
print(sk_demo["S"])
print("\nМатрица P (перестановка 7x7):")
print(sk_demo["P"])

## 5. Проверка корректности

Проверка корректности работы схемы на всех 16 возможных 4-битных сообщениях.

In [ ]:
pk, sk = keygen()
all_correct = True

print("Проверка всех 16 сообщений:")
print(f"{'Сообщение m':<20} {'Шифртекст c':<25} {'Расшифровано':<20} {'OK?'}")
print("-" * 75)

for i in range(16):
    m = np.array([(i >> 3) & 1, (i >> 2) & 1, (i >> 1) & 1, i & 1], dtype=np.int8)
    c = encrypt(m, pk)
    m_dec = decrypt(c, sk)
    ok = np.array_equal(m_dec, m)
    if not ok:
        all_correct = False
    print(f"{m!s:<20} {c!s:<25} {m_dec!s:<20} {'✓' if ok else '✗ ОШИБКА!'}")

print()
if all_correct:
    print("Проверка завершена успешно! Все 16 сообщений верно расшифрованы.")
else:
    print("ОШИБКА: Не все сообщения расшифрованы правильно!")

## 6. Атака brute-force (полный перебор)

Демонстрация небезопасности маленьких параметров.

**Идея атаки:** для каждого из $2^4 = 16$ возможных сообщений $m$ вычислить $m \cdot G'$,
затем проверить, равен ли вес Хэмминга разности $c - m \cdot G'$ единице (т.е. соответствует ли это вектору ошибки веса 1).

In [ ]:
m_original = np.array([1, 0, 1, 0], dtype=np.int8)
c = encrypt(m_original, pk)

start_time = time.perf_counter()
m_recovered, e_recovered = brute_force(c, pk)
end_time = time.perf_counter()

print(f"Исходное сообщение:      {m_original}")
print(f"Перехваченный шифртекст: {c}")
print(f"Восстановленное сообщение: {m_recovered}")
print(f"Восстановленная ошибка:  {e_recovered}")
print(f"Время атаки:             {(end_time - start_time) * 1000:.4f} мс")
print()
print("Всего перебрано сообщений: 16 (все возможные 4-битные)")
print("Это тривиально — маленькие параметры НЕ обеспечивают безопасность!")

## 7. Сравнение с Classic McEliece и HQC

Таблица размеров ключей и шифртекстов для учебной схемы и реальных постквантовых алгоритмов.

In [ ]:
print(
    f"{'Схема':<20} | {'Длина откр. ключа (бит)':<25} | "
    f"{'Длина закр. ключа (бит)':<25} | {'Шифртекст (бит)':<15}"
)
print("-" * 95)
print(f"{'Учебный Мак-Элис':<20} | {28:<25} | {107:<25} | {7:<15}")
print(f"{'Classic McEliece':<20} | {'~2 000 000':<25} | {'~100 000':<25} | {'~1000':<15}")
print(f"{'HQC-128':<20} | {18000:<25} | {18288:<25} | {35840:<15}")
print()
print("Пояснения:")
print("  Учебный Мак-Элис: n=7, k=4, t=1")
print("    - Открытый ключ G' (4x7 бит матрица) = 28 бит")
print(
    "    - Закрытый ключ: S (16 бит) + P (49 бит) + S_inv (16 бит) + P_inv (49 бит) + H (21 бит) ≈ 107 бит"
)
print("    - Шифртекст: 7 бит")
print()
print("  Classic McEliece 6960119: n=6960, k=5413, t=119")
print("    - Открытый ключ: ~1.36 МБ (~11 Мбит), закрытый ключ: ~13.9 КБ")
print()
print("  HQC-128: уровень безопасности 128 бит")
print("    - Использует квазициклические коды (не Хэмминга), иная структура ключей")